# Accuracy–Latency–Cost Pareto frontier — all four architectures

Pulls each architecture's final numbers from its saved result files and plots
which ones are **Pareto-optimal**: no other architecture beats them on every
axis at once. Answers RQ3/RQ4.

| Architecture | Notebook |
|---|---|
| Vector RAG | `pipelines/vector_rag/vector_rag_pipeline.ipynb` |
| Vectorless RAG (PageIndex) | `pipelines/vectorless_rag/vectorless_rag_pipeline.ipynb` |
| PageIndex hybrid | `pipelines/experimental/pageindex_hybrid_pipeline.ipynb` |
| Long context | `pipelines/long_context/long_context_pipeline.ipynb` |

**Read-only** — makes no API calls, only reads `experiments/results/`. Safe to
re-run any time. Runs in Colab (reads from Drive) or locally (after `git pull`,
once every pipeline has synced its results).

### How each number is defined — same rules for all four

- **Accuracy** = questions scored `Correct` ÷ all 150. A question with no
  saved score counts as not correct, so a pipeline can't look better by
  skipping hard questions.
- **Cost per 1k queries** = per-query cost × 1,000. Per-query cost is the sum,
  over the stages a query goes through, of each stage's average cost per
  question, after de-duplicating the append-only cost logs (same
  `clean_stage_costs` / `dedupe_by_last` as each pipeline's own cost cell) and
  re-pricing against `evaluation/cost_tracker.py`'s current table. The judge
  is left out (it grades answers and is not part of answering them), and so
  are latency-pass calls.
- **One-time indexing cost** is reported in the table but kept off the plot —
  it's paid once per document, not per query.
- **Latency** = median and p95 over every timed pass in each pipeline's
  dedicated latency run (10 questions × 3 repeats).
- **Borrowed stages**: the hybrid reuses vectorless's navigation and vector
  RAG's query expansion instead of re-running them, but a real query would
  still pay for them, so their cost and latency are added in. See
  `COST_STAGES` and `BORROWED_LATENCY` below; the assumptions are spelled out there.

---
## Setup

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/drive/MyDrive/financebench_project')
    !git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit
except ImportError:
    # Local: walk up from this notebook to the repo root
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data" / "financebench_open_source.jsonl").exists())

sys.path.insert(0, str(REPO_ROOT))
from evaluation.cost_tracker import clean_stage_costs, clean_doc_level_costs, dedupe_by_last, recompute_cost_usd

RESULTS_DIR = REPO_ROOT / "experiments" / "results"
FIGURES_DIR = REPO_ROOT / "analysis" / "figures"
N_QUESTIONS = 150
print(f"Reading results from {RESULTS_DIR}")

---
## Result files per architecture

`label_field` differs because the notebooks were written at different times:
vector RAG and long context save the final verdict as `score_label`,
vectorless and the hybrid as `label`.

In [ ]:
R = RESULTS_DIR

VR_COSTS = R / "vector_rag_costs.jsonl"
VL_COSTS = R / "vectorless_rag_costs.jsonl"
HY_COSTS = R / "pageindex_hybrid_expanded_costs.jsonl"
LC_COSTS = R / "long_context_costs.jsonl"

VR_LATENCY = R / "vector_rag_latency.jsonl"
VL_LATENCY = R / "vectorless_rag_latency.jsonl"
HY_LATENCY = R / "pageindex_hybrid_expanded_latency.jsonl"
LC_LATENCY = R / "long_context_latency.jsonl"

VL_NAV_EXPANDED = R / "vectorless_rag_stage_navigation_expanded.jsonl"

ARCHITECTURES = {
    "Vector RAG": {
        "scoring": R / "vector_rag_stage_scoring.jsonl", "label_field": "score_label",
        "generation": R / "vector_rag_stage_generation.jsonl", "latency": VR_LATENCY,
    },
    "Vectorless RAG": {
        "scoring": R / "vectorless_rag_stage_scoring.jsonl", "label_field": "label",
        "generation": R / "vectorless_rag_stage_generation.jsonl", "latency": VL_LATENCY,
    },
    "PageIndex hybrid": {
        "scoring": R / "pageindex_hybrid_expanded_stage_scoring.jsonl", "label_field": "label",
        "generation": R / "pageindex_hybrid_expanded_stage_generation.jsonl", "latency": HY_LATENCY,
    },
    "Long context": {
        "scoring": R / "long_context_stage_scoring.jsonl", "label_field": "score_label",
        "generation": R / "long_context_stage_generation.jsonl", "latency": LC_LATENCY,
    },
}

# Per-query stages each architecture pays for: (cost log, stage name, output file
# to match against -- or None to keep the most recent record per question, for
# stages with no per-question output file of their own).
COST_STAGES = {
    "Vector RAG": [
        (VR_COSTS, "query_expansion", None),
        (VR_COSTS, "rerank", R / "vector_rag_stage_rerank.jsonl"),        # Jina: unpriced, counted as $0
        (VR_COSTS, "selection", R / "vector_rag_stage_selection.jsonl"),
        (VR_COSTS, "generation", R / "vector_rag_stage_generation.jsonl"),
    ],
    "Vectorless RAG": [
        (VL_COSTS, "query_expansion", VL_NAV_EXPANDED),
        (VL_COSTS, "navigation_expanded", VL_NAV_EXPANDED),
        (VL_COSTS, "generation", R / "vectorless_rag_stage_generation.jsonl"),
    ],
    "PageIndex hybrid": [
        (HY_COSTS, "generation", R / "pageindex_hybrid_expanded_stage_generation.jsonl"),
        # borrowed -- see the hybrid notebook's Stage 11 for why each one counts
        (VL_COSTS, "query_expansion", VL_NAV_EXPANDED),       # fed the navigation below
        (VL_COSTS, "navigation_expanded", VL_NAV_EXPANDED),   # picks the pages hybrid search is limited to
        (VR_COSTS, "query_expansion", None),                  # the query hybrid search runs with
        # hybrid rerank is Jina too (unpriced), and isn't in HY_COSTS at all
    ],
    "Long context": [
        (LC_COSTS, "generation", R / "long_context_stage_generation.jsonl"),
    ],
}

# One-time preprocessing, doc-level. Vector RAG's Stella indexing ran locally ($0);
# long context has none. The hybrid reuses vectorless's trees.
INDEXING_STAGES = {
    "Vector RAG": [],
    "Vectorless RAG": [(VL_COSTS, "indexing_flash"), (VL_COSTS, "indexing_fallback")],
    "PageIndex hybrid": [(VL_COSTS, "indexing_flash"), (VL_COSTS, "indexing_fallback")],
    "Long context": [],
}

# Latency: stages a real query would wait for but that the pipeline's own latency
# pass didn't time. Each is added as that stage's median, taken from the pipeline
# that did time it.
#   - Vectorless's latency pass navigates on the raw question, but its real
#     pipeline expands the question first -> add one query expansion (vector RAG's
#     timing: same model, same prompt).
#   - The hybrid's pass starts from saved navigation and a saved expanded query ->
#     add navigation (vectorless) plus one expansion and the query embedding (vector
#     RAG). Only ONE expansion is added: the two expansions don't depend on each
#     other, so a real system would run them at the same time.
BORROWED_LATENCY = {
    "Vector RAG": [],
    "Vectorless RAG": [(VR_LATENCY, "query_expansion")],
    "PageIndex hybrid": [(VL_LATENCY, "navigate"), (VR_LATENCY, "query_expansion"),
                         (VR_LATENCY, "query_embedding")],
    "Long context": [],
}

---
## Compute accuracy, cost and latency

In [ ]:
def load_jsonl(path) -> list[dict]:
    if not path.exists():
        return []
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


def latest_by_id(path) -> dict:
    """One record per question -- the most recent, if a stage was re-run."""
    records = sorted(load_jsonl(path), key=lambda r: r.get("timestamp", 0))
    return {r["financebench_id"]: r for r in records}


def accuracy(spec) -> dict:
    scored = latest_by_id(spec["scoring"])
    n_correct = sum(r.get(spec["label_field"]) == "Correct" for r in scored.values())
    return {"n_scored": len(scored), "n_correct": n_correct, "accuracy": n_correct / N_QUESTIONS}


def per_query_cost(stages) -> tuple[float, list[str]]:
    """Sum over stages of that stage's mean cost per question. Returns
    (cost per query in $, names of stages with no price set)."""
    total, unpriced = 0.0, []
    for cost_path, stage, output_path in stages:
        records = load_jsonl(cost_path)
        if output_path is None:
            kept = dedupe_by_last(records, stage)
        else:
            kept, _ = clean_stage_costs(records, stage, latest_by_id(output_path))
        kept = recompute_cost_usd(kept)
        if not kept:
            print(f"    WARNING: no cost records for {stage} in {cost_path.name}")
            continue
        if any(r["cost_usd"] is None for r in kept):
            unpriced.append(stage)
        total += sum(r["cost_usd"] or 0.0 for r in kept) / len(kept)
    return total, unpriced


def indexing_cost(stages) -> float:
    total = 0.0
    for cost_path, stage in stages:
        kept, _ = clean_doc_level_costs(load_jsonl(cost_path), stage)
        total += sum(r["cost_usd"] or 0.0 for r in recompute_cost_usd(kept))
    return total


def latency(spec, borrowed) -> dict:
    runs = load_jsonl(spec["latency"])
    extra = sum(np.median([r["timings_sec"][stage] for r in load_jsonl(path)]) for path, stage in borrowed)
    totals = np.array([r["timings_sec"]["total"] for r in runs]) + extra
    return {"n_latency_runs": len(runs), "borrowed_latency_sec": extra,
            "median_latency_sec": np.median(totals), "p95_latency_sec": np.percentile(totals, 95)}


rows = []
for name, spec in ARCHITECTURES.items():
    missing = [p.name for p in (spec["scoring"], spec["generation"], spec["latency"]) if not p.exists()]
    if missing:
        print(f"SKIPPING {name} -- missing {missing} (run its notebook / sync its results first)")
        continue
    print(name)
    cost, unpriced = per_query_cost(COST_STAGES[name])
    rows.append({
        "architecture": name,
        **accuracy(spec),
        "cost_per_query_usd": cost,
        "cost_per_1k_usd": cost * 1000,
        "unpriced_stages": ", ".join(unpriced),
        "indexing_cost_usd": indexing_cost(INDEXING_STAGES[name]),
        **latency(spec, BORROWED_LATENCY[name]),
    })

results = pd.DataFrame(rows).set_index("architecture")

---
## Which architectures are Pareto-optimal?

An architecture is **dominated** if some other one is at least as good on all
three axes (accuracy up, cost down, median latency down) and strictly better on
at least one. Whatever isn't dominated is on the frontier: picking between
frontier architectures means trading one axis for another.

In [ ]:
def pareto_optimal(df, maximize=(), minimize=()) -> pd.Series:
    # Flip minimized axes so "bigger is better" everywhere
    scores = pd.concat([df[list(maximize)], -df[list(minimize)]], axis=1).to_numpy()
    optimal = []
    for i in range(len(scores)):
        dominated = any(
            (scores[j] >= scores[i]).all() and (scores[j] > scores[i]).any()
            for j in range(len(scores)) if j != i
        )
        optimal.append(not dominated)
    return pd.Series(optimal, index=df.index)


results["pareto_optimal"] = pareto_optimal(
    results, maximize=["accuracy"], minimize=["cost_per_1k_usd", "median_latency_sec"])

table = pd.DataFrame({
    "Accuracy": results["accuracy"].map("{:.1%}".format),
    "Correct / scored": results["n_correct"].astype(str) + " / " + results["n_scored"].astype(str),
    "Cost / 1k queries": results["cost_per_1k_usd"].map("${:.2f}".format),
    "Indexing (one-time)": results["indexing_cost_usd"].map("${:.2f}".format),
    "Median latency": results["median_latency_sec"].map("{:.1f}s".format),
    "p95 latency": results["p95_latency_sec"].map("{:.1f}s".format),
    "Pareto-optimal": results["pareto_optimal"].map({True: "yes", False: "no"}),
    "Unpriced stages": results["unpriced_stages"],
})
table

---
## Pareto plots

Two 2-D views of the same result: **up and to the left is better** in both.
Filled blue points are on that view's frontier, and the blue step line joins
them. Hollow grey points are dominated in that view. The frontier is
recomputed for each view, so an architecture can be on one plot's frontier and
not the other's. The table above gives the combined 3-axis answer.

Latency points sit at the median, with a thin line out to p95.
Both x-axes use a log scale: long context is roughly 10× the cost and 5× the latency of the
RAG variants, and on a linear axis the three RAG points would sit on top of each other.

In [ ]:
from matplotlib.ticker import FuncFormatter, NullFormatter

FRONTIER = "#2a78d6"   # blue: on this view's frontier
DOMINATED = "#8a8984"  # neutral grey: dominated in this view
TEXT = "#52514e"

plt.rcParams.update({
    "font.size": 10, "axes.edgecolor": "#c9c8c2", "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "axes.spines.top": False, "axes.spines.right": False,
})


def frontier_points(df, x):
    """2-D frontier for (minimize x, maximize accuracy), sorted by x."""
    on = pareto_optimal(df, maximize=["accuracy"], minimize=[x])
    return df[on].sort_values(x)


def label_points(ax, xs, ys, min_gap_pt=13):
    """Name each point, just right of it. Points close together (e.g. the three RAG
    variants) get their labels pushed apart vertically so none overlap."""
    ax.figure.canvas.draw()   # settle axis limits so data -> screen positions are final
    to_pt = 72 / ax.figure.dpi
    pts = sorted(((ax.transData.transform((x, y)) * to_pt, (x, y), name) for name, x, y in zip(xs.index, xs, ys)),
                 key=lambda p: p[0][1])
    placed = []   # (x, y) of each label so far, in points
    for (px, py), xy, name in pts:
        ly = py + 4
        for qx, qy in placed:
            if abs(qx - px) < 120 and ly - qy < min_gap_pt:
                ly = qy + min_gap_pt
        placed.append((px, ly))
        # anchored to the data point, so later layout changes move label and point together
        ax.annotate(name, xy=xy, xytext=(8, ly - py), textcoords="offset points",
                    fontsize=9, color=TEXT, va="bottom")


def draw_view(ax, x, xlabel, fmt, whisker_to=None, log_x=False):
    front = frontier_points(results, x)
    # Frontier as a step line: moving right, accuracy only steps up
    ax.step(front[x], front["accuracy"] * 100, where="post", color=FRONTIER, linewidth=2, zorder=1)

    for name, r in results.iterrows():
        on = name in front.index
        color = FRONTIER if on else DOMINATED
        if whisker_to is not None:
            ax.plot([r[x], r[whisker_to]], [r["accuracy"] * 100] * 2, color=color, linewidth=1, alpha=0.6, zorder=1)
            ax.plot(r[whisker_to], r["accuracy"] * 100, marker="|", markersize=8, color=color, alpha=0.6)
        ax.scatter(r[x], r["accuracy"] * 100, s=70, zorder=3, linewidths=2,
                   facecolors=color if on else "white", edgecolors=color)

    if log_x:
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(FuncFormatter(fmt))
        ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Accuracy (% correct of 150)")
    ax.grid(True, color="#e6e5e0", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.margins(x=0.25, y=0.15)
    label_points(ax, results[x], results["accuracy"] * 100)


fig, (ax_cost, ax_lat) = plt.subplots(1, 2, figsize=(12, 4.8))
draw_view(ax_cost, "cost_per_1k_usd", "Cost per 1,000 queries (USD, log scale)",
          fmt=lambda v, _: f"${v:g}", log_x=True)
ax_cost.set_title("Accuracy vs. cost", loc="left", fontsize=11, color="#0b0b0b")
draw_view(ax_lat, "median_latency_sec", "End-to-end latency (s, log scale): median, line to p95",
          fmt=lambda v, _: f"{v:g}s", whisker_to="p95_latency_sec", log_x=True)
ax_lat.set_title("Accuracy vs. latency", loc="left", fontsize=11, color="#0b0b0b")
fig.tight_layout()

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
for ext in ("png", "pdf"):   # pdf for the thesis (vector), png for quick viewing
    fig.savefig(FIGURES_DIR / f"pareto_frontier.{ext}", dpi=200, bbox_inches="tight")
print(f"saved -> {FIGURES_DIR}/pareto_frontier.png / .pdf")
plt.show()

---
## Save the numbers

Same numbers as the table, unformatted, for the thesis write-up or any later analysis.

In [ ]:
out_path = RESULTS_DIR / "pareto_summary.csv"
results.to_csv(out_path)
print(f"saved -> {out_path}")